# Find Excluded Trials For Latent Regression

这个 notebook 用最直接的代码形式，把行为数据和 latent 对应的 metadata 对齐，找出：

- 哪些行为 trial 被保留进了 latent 数据
- 哪些行为 trial 被排除了
- 每个保留 trial 在 latent `Z` 里的索引是多少

这里不写成函数，尽量保持每一步都能单独看懂。

# 虽然是AI写的，但是确认没问题，逻辑是：把behavior data和metadata合并，然后排除没对齐的nan值，就可以得到完整的行为数据。

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve().parents[0]

behavior_path = project_root / 'script_pre_EEG' / 'Kosciessa_et_al_2021' / 'temp_data' / 'behavior_data_all.csv'
metadata_path = project_root / 'dataset_fixed' / 'metadata.csv'
output_dir = project_root / 'script_regression' / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

print('project_root =', project_root)
print('behavior_path =', behavior_path)
print('metadata_path =', metadata_path)
print('output_dir =', output_dir)

project_root = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet
behavior_path = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_pre_EEG/Kosciessa_et_al_2021/temp_data/behavior_data_all.csv
metadata_path = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet/dataset_fixed/metadata.csv
output_dir = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression/data/processed


In [3]:
behavior = pd.read_csv(behavior_path)
metadata = pd.read_csv(metadata_path)

print('behavior shape:', behavior.shape)
print('metadata shape:', metadata.shape)

display(behavior.head())
display(metadata.head())

behavior shape: (10258, 16)
metadata shape: (7297, 28)


,Unnamed: 0,probe_accuracy,probe_rt,probe_attribute,cue_dimensionality,probe_leftrightwin,cue_color,cue_direction,cue_size,cue_luminance,rt_is_not_outlier,subj_idx,stim_onset,stim_code,resp_onset_sample,subj_id
0,0,1.0,0.72165,1,2,2,1,1,1,1,True,sub-STSWD1117,33964,8,33964.0,sub-STSWD1117
1,1,1.0,0.82361,2,2,2,1,1,1,1,True,sub-STSWD1117,36396,8,36396.0,sub-STSWD1117
2,2,0.0,1.48840,2,2,1,1,1,1,1,True,sub-STSWD1117,41260,8,41260.0,sub-STSWD1117
3,3,0.0,0.92432,2,2,1,1,1,1,1,True,sub-STSWD1117,43692,8,43692.0,sub-STSWD1117
4,4,1.0,1.19230,1,2,1,1,1,1,1,True,sub-STSWD1117,46124,8,46124.0,sub-STSWD1117


,original_row_index,probe_accuracy,probe_rt,probe_attribute,cue_dimensionality,probe_leftrightwin,cue_color,cue_direction,cue_size,cue_luminance,...,trial_id,RT_ms,correctness,condition,difficulty,evidence_strength,choice,response_hand,artifact_rejection_flag,alignment
0,0,1.0,0.72165,1,2,2,1,1,1,1,...,sub-STSWD1117_0000,721.65,1.0,1,2,2,2,2,0,response_locked
1,1,1.0,0.82361,2,2,2,1,1,1,1,...,sub-STSWD1117_0001,823.61,1.0,2,2,2,2,2,0,response_locked
2,2,0.0,1.48840,2,2,1,1,1,1,1,...,sub-STSWD1117_0002,1488.40,0.0,2,2,2,1,1,0,response_locked
3,3,0.0,0.92432,2,2,1,1,1,1,1,...,sub-STSWD1117_0003,924.32,0.0,2,2,2,1,1,0,response_locked
4,4,1.0,1.19230,1,2,1,1,1,1,1,...,sub-STSWD1117_0004,1192.30,1.0,1,2,2,1,1,0,response_locked


## Step 1: 给行为表加上被试内 trial 编号

`dataset_fixed/metadata.csv` 里保留了 `within_subject_trial_index`。
所以我们也在原始行为表里，按被试顺序重新生成同样的编号，后面就可以一一对应。

In [4]:
behavior = behavior.copy()

# 原始 csv 第一列通常会被读成 Unnamed: 0，这一列就是原始全局行号
if 'Unnamed: 0' in behavior.columns:
    behavior = behavior.rename(columns={'Unnamed: 0': 'original_row_index'})
else:
    behavior['original_row_index'] = behavior.index

# 原始行为表中的全局行号
behavior['global_behavior_row_index'] = behavior.index

# subj_idx 就是被试 ID，例如 sub-STSWD1117
behavior['subject_id'] = behavior['subj_idx']

# 在每个被试内部，从 0 开始给每个 trial 编号
behavior['within_subject_trial_index'] = behavior.groupby('subject_id').cumcount()

display(
    behavior[
        [
            'global_behavior_row_index',
            'original_row_index',
            'subject_id',
            'within_subject_trial_index',
            'probe_accuracy',
            'probe_rt',
        ]
    ].head(12)
)

,global_behavior_row_index,original_row_index,subject_id,within_subject_trial_index,probe_accuracy,probe_rt
0,0,0,sub-STSWD1117,0,1.0,0.72165
1,1,1,sub-STSWD1117,1,1.0,0.82361
2,2,2,sub-STSWD1117,2,0.0,1.48840
3,3,3,sub-STSWD1117,3,0.0,0.92432
4,4,4,sub-STSWD1117,4,1.0,1.19230
5,5,5,sub-STSWD1117,5,1.0,1.21420
6,6,6,sub-STSWD1117,6,1.0,0.93956
7,7,7,sub-STSWD1117,7,1.0,0.61344
8,8,8,sub-STSWD1117,8,1.0,0.35636
9,9,9,sub-STSWD1117,9,1.0,0.55931


## Step 2: 给 latent metadata 加上 latent trial 索引

latent 的 trial 顺序就是 `metadata.csv` 的行顺序，所以这里直接把行号当作 `latent_trial_index`。

In [5]:
metadata = metadata.copy()
metadata['latent_trial_index'] = metadata.index

display(
    metadata[
        [
            'latent_trial_index',
            'subject_id',
            'within_subject_trial_index',
            'trial_id',
            'original_row_index',
        ]
    ].head(12)
)

,latent_trial_index,subject_id,within_subject_trial_index,trial_id,original_row_index
0,0,sub-STSWD1117,0,sub-STSWD1117_0000,0
1,1,sub-STSWD1117,1,sub-STSWD1117_0001,1
2,2,sub-STSWD1117,2,sub-STSWD1117_0002,2
3,3,sub-STSWD1117,3,sub-STSWD1117_0003,3
4,4,sub-STSWD1117,4,sub-STSWD1117_0004,4
5,5,sub-STSWD1117,5,sub-STSWD1117_0005,5
6,6,sub-STSWD1117,6,sub-STSWD1117_0006,6
7,7,sub-STSWD1117,7,sub-STSWD1117_0007,7
8,8,sub-STSWD1117,9,sub-STSWD1117_0009,9
9,9,sub-STSWD1117,11,sub-STSWD1117_0011,11


## Step 3: 用 subject_id + within_subject_trial_index 对齐

这两个字段组合起来，可以唯一标识“某个被试的第几个 trial”。

对齐后：
- 如果某个行为 trial 能在 metadata 里找到，就说明它进入了 latent 数据
- 如果找不到，就说明它被排除了

In [6]:
alignment = behavior.merge(
    metadata[
        [
            'subject_id',
            'within_subject_trial_index',
            'latent_trial_index',
            'trial_id',
            'original_row_index',
        ]
    ].rename(columns={'original_row_index': 'metadata_original_row_index'}),
    on=['subject_id', 'within_subject_trial_index'],
    how='left'
)

alignment['in_latents'] = alignment['latent_trial_index'].notna().astype(int)

display(
    alignment[
        [
            'global_behavior_row_index',
            'subject_id',
            'within_subject_trial_index',
            'in_latents',
            'latent_trial_index',
            'trial_id',
            'probe_accuracy',
            'probe_rt',
        ]
    ].head(20)
)

,global_behavior_row_index,subject_id,within_subject_trial_index,in_latents,latent_trial_index,trial_id,probe_accuracy,probe_rt
0,0,sub-STSWD1117,0,1,0.0,sub-STSWD1117_0000,1.0,0.72165
1,1,sub-STSWD1117,1,1,1.0,sub-STSWD1117_0001,1.0,0.82361
2,2,sub-STSWD1117,2,1,2.0,sub-STSWD1117_0002,0.0,1.48840
3,3,sub-STSWD1117,3,1,3.0,sub-STSWD1117_0003,0.0,0.92432
4,4,sub-STSWD1117,4,1,4.0,sub-STSWD1117_0004,1.0,1.19230
5,5,sub-STSWD1117,5,1,5.0,sub-STSWD1117_0005,1.0,1.21420
6,6,sub-STSWD1117,6,1,6.0,sub-STSWD1117_0006,1.0,0.93956
7,7,sub-STSWD1117,7,1,7.0,sub-STSWD1117_0007,1.0,0.61344
8,8,sub-STSWD1117,8,0,NaN,NaN,1.0,0.35636
9,9,sub-STSWD1117,9,1,8.0,sub-STSWD1117_0009,1.0,0.55931


## Step 4: 分出保留的 trial 和被排除的 trial

In [8]:
retained_trials = alignment[alignment['in_latents'] == 1].copy()
missing_trials = alignment[alignment['in_latents'] == 0].copy()

print('behavior total trials      :', len(alignment))
print('retained trials in latents :', len(retained_trials))
print('excluded trials            :', len(missing_trials))

display(retained_trials.head())
display(missing_trials.head())

behavior total trials      : 10258
retained trials in latents : 7297
excluded trials            : 2961


,original_row_index,probe_accuracy,probe_rt,probe_attribute,cue_dimensionality,probe_leftrightwin,cue_color,cue_direction,cue_size,cue_luminance,...,stim_code,resp_onset_sample,subj_id,global_behavior_row_index,subject_id,within_subject_trial_index,latent_trial_index,trial_id,metadata_original_row_index,in_latents
0,0,1.0,0.72165,1,2,2,1,1,1,1,...,8,33964.0,sub-STSWD1117,0,sub-STSWD1117,0,0.0,sub-STSWD1117_0000,0.0,1
1,1,1.0,0.82361,2,2,2,1,1,1,1,...,8,36396.0,sub-STSWD1117,1,sub-STSWD1117,1,1.0,sub-STSWD1117_0001,1.0,1
2,2,0.0,1.48840,2,2,1,1,1,1,1,...,8,41260.0,sub-STSWD1117,2,sub-STSWD1117,2,2.0,sub-STSWD1117_0002,2.0,1
3,3,0.0,0.92432,2,2,1,1,1,1,1,...,8,43692.0,sub-STSWD1117,3,sub-STSWD1117,3,3.0,sub-STSWD1117_0003,3.0,1
4,4,1.0,1.19230,1,2,1,1,1,1,1,...,8,46124.0,sub-STSWD1117,4,sub-STSWD1117,4,4.0,sub-STSWD1117_0004,4.0,1


,original_row_index,probe_accuracy,probe_rt,probe_attribute,cue_dimensionality,probe_leftrightwin,cue_color,cue_direction,cue_size,cue_luminance,...,stim_code,resp_onset_sample,subj_id,global_behavior_row_index,subject_id,within_subject_trial_index,latent_trial_index,trial_id,metadata_original_row_index,in_latents
8,8,1.0,0.35636,2,1,2,1,1,1,1,...,8,57904.0,sub-STSWD1117,8,sub-STSWD1117,8,NaN,NaN,NaN,0
10,10,1.0,0.36126,2,1,2,1,1,1,1,...,8,62768.0,sub-STSWD1117,10,sub-STSWD1117,10,NaN,NaN,NaN,0
12,12,1.0,0.30308,2,1,2,1,1,1,1,...,8,67632.0,sub-STSWD1117,12,sub-STSWD1117,12,NaN,NaN,NaN,0
13,13,1.0,0.37306,2,1,1,1,1,1,1,...,8,70064.0,sub-STSWD1117,13,sub-STSWD1117,13,NaN,NaN,NaN,0
14,14,1.0,0.33796,2,1,2,1,1,1,1,...,8,72496.0,sub-STSWD1117,14,sub-STSWD1117,14,NaN,NaN,NaN,0


## Step 5: 看被排除 trial 的可能原因

这里只根据行为表本身能看出来的情况先分一层：

- `probe_rt` 和 `probe_accuracy` 缺失
- 其他情况大概率就是 EEG response-locked window 里有 `NaN/inf`，所以没进入 `dataset_fixed`

注意：真正的 EEG 非有限值位置不在这个表里，这里只是做原因标记。

In [9]:
missing_trials['exclusion_reason'] = 'likely_nonfinite_eeg_window'

mask_missing_behavior = (
    missing_trials['probe_rt'].isna()
    | missing_trials['probe_accuracy'].isna()
)

missing_trials.loc[mask_missing_behavior, 'exclusion_reason'] = 'missing_rt_or_accuracy'

display(missing_trials['exclusion_reason'].value_counts())
display(
    missing_trials[
        [
            'global_behavior_row_index',
            'subject_id',
            'within_subject_trial_index',
            'probe_accuracy',
            'probe_rt',
            'exclusion_reason',
        ]
    ].head(20)
)

exclusion_reason
likely_nonfinite_eeg_window    2927
missing_rt_or_accuracy           34
Name: count, dtype: int64

,global_behavior_row_index,subject_id,within_subject_trial_index,probe_accuracy,probe_rt,exclusion_reason
8,8,sub-STSWD1117,8,1.0,0.35636,likely_nonfinite_eeg_window
10,10,sub-STSWD1117,10,1.0,0.36126,likely_nonfinite_eeg_window
12,12,sub-STSWD1117,12,1.0,0.30308,likely_nonfinite_eeg_window
13,13,sub-STSWD1117,13,1.0,0.37306,likely_nonfinite_eeg_window
14,14,sub-STSWD1117,14,1.0,0.33796,likely_nonfinite_eeg_window
24,24,sub-STSWD1117,24,NaN,NaN,missing_rt_or_accuracy
39,39,sub-STSWD1117,39,1.0,0.47802,likely_nonfinite_eeg_window
48,48,sub-STSWD1117,48,1.0,0.25835,likely_nonfinite_eeg_window
49,49,sub-STSWD1117,49,1.0,0.41527,likely_nonfinite_eeg_window
50,50,sub-STSWD1117,50,1.0,0.46321,likely_nonfinite_eeg_window


## Step 6: 导出结果

后面你做回归时最常用的是这三个表：

- `behavior_latent_alignment.csv`：总对齐表
- `retained_trials_for_latents.csv`：进入 latent 的行为 trial
- `missing_trials_from_latents.csv`：被排除的行为 trial

In [10]:
alignment.to_csv(output_dir / 'behavior_latent_alignment.csv', index=False)
retained_trials.to_csv(output_dir / 'retained_trials_for_latents.csv', index=False)
missing_trials.to_csv(output_dir / 'missing_trials_from_latents.csv', index=False)

print('saved:')
print(output_dir / 'behavior_latent_alignment.csv')
print(output_dir / 'retained_trials_for_latents.csv')
print(output_dir / 'missing_trials_from_latents.csv')

saved:
/Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression/data/processed/behavior_latent_alignment.csv
/Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression/data/processed/retained_trials_for_latents.csv
/Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression/data/processed/missing_trials_from_latents.csv


## Step 7: 回归前怎么用

如果你后面要把行为和 latent `Z` 做回归，最简单的对齐方式就是：

1. 用 `retained_trials` 或 `alignment[alignment['in_latents'] == 1]` 作为行为子集
2. 按 `latent_trial_index` 去取 `Z[latent_trial_index]`

这样行为行和 latent trial 就是一一对应的。

In [11]:
regression_ready_behavior = retained_trials.sort_values('latent_trial_index').reset_index(drop=True)

display(
    regression_ready_behavior[
        [
            'latent_trial_index',
            'subject_id',
            'within_subject_trial_index',
            'trial_id',
            'probe_accuracy',
            'probe_rt',
        ]
    ].head(20)
)

print('regression_ready_behavior shape =', regression_ready_behavior.shape)

,latent_trial_index,subject_id,within_subject_trial_index,trial_id,probe_accuracy,probe_rt
0,0.0,sub-STSWD1117,0,sub-STSWD1117_0000,1.0,0.72165
1,1.0,sub-STSWD1117,1,sub-STSWD1117_0001,1.0,0.82361
2,2.0,sub-STSWD1117,2,sub-STSWD1117_0002,0.0,1.48840
3,3.0,sub-STSWD1117,3,sub-STSWD1117_0003,0.0,0.92432
4,4.0,sub-STSWD1117,4,sub-STSWD1117_0004,1.0,1.19230
5,5.0,sub-STSWD1117,5,sub-STSWD1117_0005,1.0,1.21420
6,6.0,sub-STSWD1117,6,sub-STSWD1117_0006,1.0,0.93956
7,7.0,sub-STSWD1117,7,sub-STSWD1117_0007,1.0,0.61344
8,8.0,sub-STSWD1117,9,sub-STSWD1117_0009,1.0,0.55931
9,9.0,sub-STSWD1117,11,sub-STSWD1117_0011,1.0,0.53218


regression_ready_behavior shape = (7297, 23)
